In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot
from sklearn.model_selection import train_test_split

In [4]:
#cargar dataset
df = pd.read_csv('./datasets/bike-sharing-demand/train.csv')

print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nPrimeras 5 filas:")
pd.set_option('display.max_columns', None)
print(df.head())
print(f"\nTipos de datos:")
print(df.dtypes)

Filas:    10886
Columnas: 12

Columnas: ['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count']

Primeras 5 filas:
              datetime  season  holiday  workingday  weather  temp   atemp  \
0  2011-01-01 00:00:00       1        0           0        1  9.84  14.395   
1  2011-01-01 01:00:00       1        0           0        1  9.02  13.635   
2  2011-01-01 02:00:00       1        0           0        1  9.02  13.635   
3  2011-01-01 03:00:00       1        0           0        1  9.84  14.395   
4  2011-01-01 04:00:00       1        0           0        1  9.84  14.395   

   humidity  windspeed  casual  registered  count  
0        81        0.0       3          13     16  
1        80        0.0       8          32     40  
2        80        0.0       5          27     32  
3        75        0.0       3          10     13  
4        75        0.0       0           1      1  

Tipos de datos:
datetime 

In [5]:
# Convertir datetime una sola vez
dt = pd.to_datetime(df['datetime'])

# Extraer componentes útiles
df['hour']      = dt.dt.hour        # hora del día (0-23)
df['month']     = dt.dt.month       # mes del año (1-12)
df['dayofweek'] = dt.dt.dayofweek   # día de la semana (0=lunes)

# Eliminar columnas
# datetime   → ya extraímos la información relevante
# casual     → data leakage, ya está incluida en count
# registered → data leakage, ya está incluida en count
df = df.drop(columns=['datetime', 'casual', 'registered'])

print(f"Columnas después del preprocesamiento:")
print(df.columns.tolist())
print(f"\nShape: {df.shape}")
print(f"\nPrimeras 5 filas:")
print(df.head())

Columnas después del preprocesamiento:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'count', 'hour', 'month', 'dayofweek']

Shape: (10886, 12)

Primeras 5 filas:
   season  holiday  workingday  weather  temp   atemp  humidity  windspeed  \
0       1        0           0        1  9.84  14.395        81        0.0   
1       1        0           0        1  9.02  13.635        80        0.0   
2       1        0           0        1  9.02  13.635        80        0.0   
3       1        0           0        1  9.84  14.395        75        0.0   
4       1        0           0        1  9.84  14.395        75        0.0   

   count  hour  month  dayofweek  
0     16     0      1          5  
1     40     1      1          5  
2     32     2      1          5  
3     13     3      1          5  
4      1     4      1          5  


In [7]:
#Valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())

# Este dataset no tiene nulos pero verificamos por completitud
nulos_total = df.isnull().sum().sum()
print(f"\nTotal de nulos: {nulos_total}")

Valores nulos por columna:
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
count         0
hour          0
month         0
dayofweek     0
dtype: int64

Total de nulos: 0


In [8]:
#Variabls númericas y categóricas
numericas   = df.select_dtypes(include='number').columns.tolist()
categoricas = df.select_dtypes(include='object').columns.tolist()

print(f"Variables numéricas ({len(numericas)}):")
print(numericas)
print(f"\nVariables categóricas ({len(categoricas)}):")
print(categoricas)

# Ver distribución de variables categóricas tipo código
print(f"\nDistribución de season:")
print(df['season'].value_counts().sort_index())
print(f"\nDistribución de weather:")
print(df['weather'].value_counts().sort_index())
print(f"\nDistribución de holiday:")
print(df['holiday'].value_counts())
print(f"\nDistribución de workingday:")
print(df['workingday'].value_counts())

Variables numéricas (12):
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'count', 'hour', 'month', 'dayofweek']

Variables categóricas (0):
[]

Distribución de season:
season
1    2686
2    2733
3    2733
4    2734
Name: count, dtype: int64

Distribución de weather:
weather
1    7192
2    2834
3     859
4       1
Name: count, dtype: int64

Distribución de holiday:
holiday
0    10575
1      311
Name: count, dtype: int64

Distribución de workingday:
workingday
1    7412
0    3474
Name: count, dtype: int64


In [9]:
# Variable objetivo
y = df['count'].values.astype(np.float32)
X = df.drop(columns=['count']).values.astype(np.float32)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split primero — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar con datos de train
def featureNormalize(X):
    mu    = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    return (X - mu) / sigma, mu, sigma

X_train_norm, mu, sigma = featureNormalize(X_train)
X_test_norm  = (X_test - mu) / sigma

print(f"\nEntrenamiento: {X_train_norm.shape}")
print(f"Prueba:        {X_test_norm.shape}")
print(f"\nMedia X_train_norm (debe ser ~0): {X_train_norm.mean():.6f}")
print(f"Std X_train_norm  (debe ser ~1): {X_train_norm.std():.6f}")

X shape: (10886, 11)
y shape: (10886,)

Entrenamiento: (8708, 11)
Prueba:        (2178, 11)

Media X_train_norm (debe ser ~0): 0.000003
Std X_train_norm  (debe ser ~1): 0.999998
